# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables** in a `.env` file:

~~`AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.~~ *(cách cũ — deprecated)*  
~~`AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.~~ *(cách cũ — deprecated)*

> ⚠️ **Lưu ý quan trọng (cập nhật 2025–2026):** Microsoft đã chuyển từ Azure AI sang Microsoft Foundry, kéo theo thay đổi toàn bộ tên biến môi trường và cách lấy endpoint:
> - `AZURE_AI_PROJECT_ENDPOINT` → **`FOUNDRY_PROJECT_ENDPOINT`**
> - `AZURE_AI_MODEL_DEPLOYMENT_NAME` → **`FOUNDRY_MODEL`**
>
> **Cách lấy `FOUNDRY_PROJECT_ENDPOINT`:**  
> Vào [Microsoft Foundry Portal](https://ai.azure.com) → Project Overview → mục **"Endpoints and keys"** → **"Azure AI model inference endpoint"**.  
> Bỏ `/models` ở cuối, endpoint có dạng:
> ```
> https://<resource-name>.services.ai.azure.com
> ```
> Ví dụ: `https://nphuc-moa27j1p-eastus2.services.ai.azure.com`

The cell below installs the Python packages you need.

In [16]:
# Cài đặt các package cần thiết
# agent-framework-foundry thay thế agent-framework-azure từ 2026
%pip install agent-framework agent-framework-foundry azure-identity python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# CÁCH CŨ (deprecated — giữ lại để tham khảo, KHÔNG chạy)
# ============================================================
# import logging
# logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)
#
# import os
# import asyncio
# from typing import Annotated
#
# from agent_framework import tool
# from agent_framework.azure import AzureAIProjectAgentProvider  # ← đã bị xoá
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# ============================================================

# ============================================================
# CÁCH MỚI (agent-framework >= 1.0.0, 2026)
# ============================================================
import os
from dotenv import load_dotenv
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient  # ← dùng foundry thay vì azure
from azure.identity import DefaultAzureCredential

load_dotenv()  # đọc biến môi trường từ file .env

print(f"   FOUNDRY_PROJECT_ENDPOINT: {os.getenv('FOUNDRY_PROJECT_ENDPOINT')}")
print(f"   FOUNDRY_MODEL: {os.getenv('FOUNDRY_MODEL')}")

credential = DefaultAzureCredential()

   FOUNDRY_PROJECT_ENDPOINT: https://nphuc-moa27j1p-eastus2.services.ai.azure.com
   FOUNDRY_MODEL: gpt-4.1-mini


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [19]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
# ============================================================
# CÁCH CŨ (deprecated — giữ lại để tham khảo, KHÔNG chạy)
# ============================================================
# agent = await provider.create_agent(
#     tools=[get_destinations],
#     name="TravelAgent",
#     instructions=(
#         "You are a helpful travel agent. Help users find their perfect vacation "
#         "destination based on their preferences. Use the get_destinations tool "
#         "to see available destinations."
#     ),
# )
#
# response = await agent.run(
#     "I'm looking for a warm beach destination. What do you recommend?"
# )
# print(response)
# ============================================================

# ============================================================
# CÁCH MỚI (agent-framework >= 1.0.0, 2026)
# ============================================================
# Thay đổi chính:
#   - Không còn dùng AzureAIProjectAgentProvider
#   - Dùng FoundryChatClient với project_endpoint (không phải connection string)
#   - Tham số deployment_name → model

agent = FoundryChatClient(
    credential=credential,
    project_endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
    model=os.getenv("FOUNDRY_MODEL"),
).as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
 )
print(response.text)

For a warm beach destination, I would recommend Barcelona, Rio de Janeiro, or Bali. These locations are known for their beautiful beaches and warm climate. Would you like more information about any of these destinations?


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [ ]:
# ============================================================
# CÁCH CŨ (deprecated — giữ lại để tham khảo, KHÔNG chạy)
# ============================================================
# async for chunk in agent.run(
#     "Tell me about Tokyo as a travel destination", stream=True
# ):
#     print(chunk, end="", flush=True)
# ============================================================

# ============================================================
# CÁCH MỚI (agent-framework >= 1.0.0, 2026)
# ============================================================
# Thay đổi chính:
#   - chunk bây giờ là AgentResponseUpdate object, cần dùng chunk.text
#     thay vì print(chunk) trực tiếp

async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    if chunk.text:
        print(chunk.text, end="", flush=True)

Tokyo is the bustling capital city of Japan and a fascinating travel destination that seamlessly blends the ultramodern with the traditional. Here are some highlights about Tokyo for travelers:

1. Culture and History: Tokyo boasts historic sites such as the Meiji Shrine, Senso-ji Temple in Asakusa, and the Imperial Palace. You can immerse yourself in traditional Japanese culture through tea ceremonies, sumo tournaments, and festivals.

2. Shopping: From luxury shopping in Ginza to quirky fashion in Harajuku, Tokyo offers an incredible variety of shopping experiences. Don’t miss the electronic district in Akihabara for gadgets and anime merchandise.

3. Food: Tokyo is a food lover's paradise, with countless Michelin-starred restaurants, sushi bars, ramen shops, and street food vendors. You can savor everything from high-end kaiseki meals to casual bites.

4. Entertainment and Nightlife: The city features vibrant nightlife with karaoke bars, izakayas, nightclubs, and themed cafes. There

## Summary

In this lesson you learned how to:

- **Create a Foundry chat client** that connects to Azure AI Foundry using `FoundryChatClient` (từ `agent_framework.foundry`).
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Create and run an agent** using `.as_agent(...)` và `agent.run(...)` rồi dùng `response.text` để lấy kết quả.
- **Stream responses** for real-time output — dùng `chunk.text` thay vì `chunk` trực tiếp.

### Bảng tóm tắt thay đổi API (cũ → mới)

| Thành phần | Cách cũ (deprecated) | Cách mới (2026) |
|---|---|---|
| Package | `agent_framework.azure` | `agent_framework.foundry` |
| Client class | `AzureAIProjectAgentProvider` | `FoundryChatClient` |
| Env var endpoint | `AZURE_AI_PROJECT_ENDPOINT` | `FOUNDRY_PROJECT_ENDPOINT` |
| Env var model | `AZURE_AI_MODEL_DEPLOYMENT_NAME` | `FOUNDRY_MODEL` |
| Tham số model | `deployment_name=` | `model=` |
| Tạo agent | `provider.create_agent(...)` | `client.as_agent(...)` |
| Lấy kết quả | `print(response)` | `print(response.text)` |
| Streaming chunk | `print(chunk)` | `print(chunk.text)` |

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.